#### Task 1: Simple vector embedding generation

**Objective:**
Generate vector embeddings from text data.

**Task Description:**

- load embedding model into ollama:
    1. Attach ollama container shell
    2. Run command in container `ollama pull pull bge-m3` or send an api call `requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False}` via the requests library to download embedding model.
- embed simple text queries

How to select the right embedding model: [MTEB - Massive Text Embedding Benchmark](https://huggingface.co/blog/mteb)

**Useful links:**

- [Ollama REST API](https://www.postman.com/postman-student-programs/ollama-api/documentation/suc47x8/ollama-rest-api)
- [Langchain Ollama Embeddings](https://python.langchain.com/docs/integrations/text_embedding/ollama/)
- [Langchain Chroma](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/)


In [1]:
import requests

In [2]:
# Download embedding model into ollama container
requests.post("http://localhost:11434/api/pull", json = {"name": "bge-m3",  "stream": False})

<Response [200]>

In [3]:
# List all available models in the ollama container
response = requests.get("http://localhost:11434/api/ps")
response.json()

{'models': []}

In [4]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

In [5]:
text = "This is a test document."

# ADD HERE YOUR CODE
# Perform vector search
query_vector = embedding_model.embed_query(text)

print(f"Embedding vector length: {len(query_vector)}")
print(query_vector[:10])

Embedding vector length: 1024
[-0.049998682, 0.021091571, -0.057094406, -0.005516551, -0.006297905, -0.02841459, 0.045169096, 0.04271729, -0.00022960188, 0.013049353]


#### Task 2: Generate embedding vectors with custom dataset

**Objective:**
Load custom dataset, preprocess it and generate vector embeddings.

**Task Description:**

- load pdf document "AI_Book.pdf" via langchain document loader: ``PyPDFLoader``
- use RecursiveCharacterTextSplitter to split documents into chunks
- generate embeddings for single documents

**RecursiveCharacterTextSplitter:**

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is `["\n\n", "\n", " ", ""]`. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

**Useful links:**

- [Langchain PyPDFLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfloader/)
- [Langchain RecursiveCharacterTextSplitter](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

pdf_doc = "./AI_Book.pdf"

# Create pdf data loader
# ADD HERE YOUR CODE
loader = PyPDFLoader(pdf_doc)

# Load and split documents in chunks
# ADD HERE YOUR CODE
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

pages_chunked = text_splitter.split_documents(pages)

print(pages_chunked[0])

# Function to clean text by removing invalid unicode characters, including surrogate pairs
def clean_text(text):
    # Remove surrogate pairs
    text = re.sub(r'[\ud800-\udfff]', '', text)
    # Optionally remove non-ASCII characters (depends on your use case)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

pages_chunked_cleaned = [
    Document(page_content=clean_text(doc.page_content), metadata=doc.metadata)
    for doc in pages_chunked
]

print(pages_chunked_cleaned[0])


C:\Users\Christina Dimovic\AppData\Local\Temp\ipykernel_29836\220114304.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


page_content='Aurélien Géron
Hands-on Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2019-05-07T15:51:31+00:00', 'moddate': '2019-05-07T15:51:31+00:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': './AI_Book.pdf', 'total_pages': 510, 'page': 2, 'page_label': '1'}
page_content='Aurlien Gron
Hands-on Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing' metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creat

In [7]:
print(pages_chunked_cleaned[1])

page_content='978-1-492-03264-9
[LSI]
Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow
by Aurlien Gron
Copyright  2019 Aurlien Gron. All rights reserved.
Printed in the United States of America.
Published by OReilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
OReilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles (http://oreilly.com). For more information, contact our corporate/institutional
sales department: 800-998-9938 or corporate@oreilly.com.
Editor: Nicole Tache
Interior Designer: David Futato
Cover Designer: Karen Montgomery
Illustrator: Rebecca Demarest
June 2019:  Second Edition
Revision History for the Early Release
2018-11-05: First Release
2019-01-24: Second Release
2019-03-07: Third Release
2019-03-29: Fourth Release
2019-04-22: Fifth Release
See http://oreilly.com/catalog/errata.csp?isbn=9781492032649 for release details.' metadata={'producer': 'Antenna

In [8]:
print(f"Number of text chunks: {len(pages_chunked_cleaned)}")

Number of text chunks: 1340


#### Task 3: Store vector embeddings from pdf document to ChromaDB vector database.

**Objective:**

Store vector embeddings into ChromaDB to store knowledge.

**Task Description:**

- create chromadb client
- create chromadb collection
- create langchain chroma db client
- store text document chunks and vector embeddings to vector databases

**Useful links:**

- [Langchain How To](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#initialization-from-client)

In [10]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

collection_name = "ai_model_book"

# ADD HERE YOUR CODE
# Create a collection
collection = client.get_or_create_collection(name=collection_name)


# ADD HERE YOUR CODE
# Create chromadb
vector_db_from_client = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding_model
)

In [11]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(pages_chunked_cleaned))]

# ADD HERE YOUR CODE
vector_db_from_client.add_documents(
    documents=pages_chunked_cleaned,
    ids=uuids
    ) # that can take a while...

['11e22e33-5b91-4e36-9477-aaac9c4cb86a',
 '0df9a830-6070-433a-bfc1-511008cf2eab',
 '4e9b1845-4658-4d80-8b26-1f555d54e365',
 'd5d61a3f-4bc5-40dc-88d4-114cfb090a24',
 '3f926cd3-2e9f-4ef0-9459-3669b6f7d3c0',
 '624b8d94-049f-4ac4-84d4-95039cf8f7b8',
 '04d89e79-7ba2-49b3-9dbb-d653410e8f19',
 'f69fa5b9-d249-4751-98bd-e78b35d7e702',
 'ca2b9a35-27ea-43ad-8d25-a6b571dc1902',
 '18bbaf8f-5b5d-40b4-a2e5-36ad17dde58c',
 '50d10cca-a1f7-4d9f-a23a-d15f2fa61074',
 'fe2590cc-cf3f-41bd-9b25-bc7e7ea48502',
 '3393e9dd-426f-4175-82af-424636ddb10e',
 '0bdcb517-e0c1-45a6-99f7-ee8d3faef16b',
 'fd39ec7f-3cba-4d77-ab83-d9efb31a80c7',
 '741b538d-b5a4-46ed-9ff2-0ebcb0967f93',
 '0fcdc9e2-1d9f-4f7b-af4b-dfdbb77e72c9',
 '5a796fba-7881-438c-81b9-a4753d3da178',
 '8b4f1915-ab12-44aa-8382-316c76487605',
 '997208ff-65b1-44c5-9c5e-b1f95f4cd7fe',
 'c623f66c-b141-4c5a-a5f4-bc739f20ed42',
 '02656555-de25-43a3-9dec-c5c3d1d4e680',
 'ef2e2fce-c091-4588-9ba8-8d65fd5e1399',
 'f6f1b845-67f0-458c-bb35-580bc774af0b',
 '996ce751-10d5-

In [12]:
client.count_collections()

1

In [ ]:
# client.delete_collection("ai_model_book")

In [13]:
# Get and check the content of the collection
collection.get(include=["documents", "metadatas", "embeddings"])

{'ids': ['11e22e33-5b91-4e36-9477-aaac9c4cb86a',
  '0df9a830-6070-433a-bfc1-511008cf2eab',
  '4e9b1845-4658-4d80-8b26-1f555d54e365',
  'd5d61a3f-4bc5-40dc-88d4-114cfb090a24',
  '3f926cd3-2e9f-4ef0-9459-3669b6f7d3c0',
  '624b8d94-049f-4ac4-84d4-95039cf8f7b8',
  '04d89e79-7ba2-49b3-9dbb-d653410e8f19',
  'f69fa5b9-d249-4751-98bd-e78b35d7e702',
  'ca2b9a35-27ea-43ad-8d25-a6b571dc1902',
  '18bbaf8f-5b5d-40b4-a2e5-36ad17dde58c',
  '50d10cca-a1f7-4d9f-a23a-d15f2fa61074',
  'fe2590cc-cf3f-41bd-9b25-bc7e7ea48502',
  '3393e9dd-426f-4175-82af-424636ddb10e',
  '0bdcb517-e0c1-45a6-99f7-ee8d3faef16b',
  'fd39ec7f-3cba-4d77-ab83-d9efb31a80c7',
  '741b538d-b5a4-46ed-9ff2-0ebcb0967f93',
  '0fcdc9e2-1d9f-4f7b-af4b-dfdbb77e72c9',
  '5a796fba-7881-438c-81b9-a4753d3da178',
  '8b4f1915-ab12-44aa-8382-316c76487605',
  '997208ff-65b1-44c5-9c5e-b1f95f4cd7fe',
  'c623f66c-b141-4c5a-a5f4-bc739f20ed42',
  '02656555-de25-43a3-9dec-c5c3d1d4e680',
  'ef2e2fce-c091-4588-9ba8-8d65fd5e1399',
  'f6f1b845-67f0-458c-bb35-

#### Task 4: Access ChromaDB and perform vector search

**Objective:**

Use query to perform vector search against ChromaDB vector database

**Task Description:**

- define query
- run vector search
- print k=3 most similar documents


**Useful links:**

- [Langchain Query ChromaDB](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma/#search-by-vector)

In [14]:
search_query = "Types of Machine Learning Systems"

results = vector_db_from_client.similarity_search(
    query=search_query,
    k=3
    )

for res in results:
    print(res.page_content)
    print(res.metadata)
    print("\n")

Types of Machine Learning Systems
There are so many different types of Machine Learning systems that it is useful to
classify them in broad categories based on:
 Whether or not they are trained with human supervision (supervised, unsuper
vised, semisupervised, and Reinforcement Learning)
 Whether or not they can learn incrementally on the fly (online versus batch
learning)
 Whether they work by simply comparing new data points to known data points,
or instead detect patterns in the training data and build a predictive model, much
like scientists do (instance-based versus model-based learning)
These criteria are not exclusive; you can combine them in any way you like. For
example, a state-of-the-art spam filter may learn on the fly using a deep neural net
work model trained using examples of spam and ham; this makes it an online, model-
based, supervised learning system.
Lets look at each of these criteria a bit more closely.
Supervised/Unsupervised Learning
{'page': 33, 'title': 'Hands